# RBL-4 Full-Run Analysis (Week 8)
**Group:** SE1916-G2 · **Owner:** MS — Nguyen Le Thuan (SE190305)

Loads `results/stats/summary.json` (produced by `scripts/compute_metric.py`, the single statistical code path per proposal §5.6) and states the reject / fail-to-reject H0 call per RQ.

α = 0.05 per RQ (pre-registered); Holm within RQ2 post-hoc; Bonferroni across the 3 RQ families reported as a stricter secondary reading (α_adj ≈ 0.017).

In [1]:
import glob, json, os
import pandas as pd
import matplotlib.pyplot as plt

# works both when executed from repo root (nbclient) and when opened from results/ in Jupyter
RAW = os.path.join('..', 'results', 'raw') if os.path.basename(os.getcwd()) == 'results' else os.path.join('results', 'raw')
EDGE = {'negative', 'boundary', 'errorcode'}
pd.set_option('display.width', 160)


In [2]:
STATS = os.path.join(os.path.dirname(RAW), 'stats', 'summary.json')
s = json.load(open(STATS, encoding='utf-8'))
print('sections:', list(s))

sections: ['RQ1_endpoint_coverage', 'RQ3_edge_cases_llm_vs_manual', 'RQ2_fault_detection', 'multiplicity']


## RQ1 — Endpoint coverage (H1: LLM coverage > 90%)

In [3]:
rq1 = s['RQ1_endpoint_coverage']
print(f"coverage = {rq1['coverage_overall_pct']}% ({rq1['ops_covered']}/{rq1['n_operations']} ops)")
print(f"one-sample Wilcoxon vs 0.90 (one-tailed): W={rq1['wilcoxon_W']}, p={rq1['p_value_one_sided']:.3e}")
print(f"effect size (rank-biserial) = {rq1['rank_biserial']}")
verdict = 'REJECT H0' if rq1['reject_H0_(coverage>90%)'] else 'FAIL TO REJECT H0'
print(f"==> RQ1: {verdict} — LLM endpoint coverage exceeds 90% (n={rq1['n_operations']})")

coverage = 100.0% (35/35 ops)
one-sample Wilcoxon vs 0.90 (one-tailed): W=630.0, p=1.649e-09
effect size (rank-biserial) = 1.0
==> RQ1: REJECT H0 — LLM endpoint coverage exceeds 90% (n=35)


## RQ2 — Fault-detection Recall (3 arms, N=133 pre-seeded mutants)

In [4]:
rq2 = s['RQ2_fault_detection']
print('overall recall:', rq2['overall_recall'])
print(f"Friedman: chi2={rq2['friedman_chi2']}, p={rq2['friedman_p']}")
print('Holm-corrected pairwise Wilcoxon:', json.dumps(rq2['pairwise_wilcoxon_holm'], indent=1))
print('Cliff\'s delta:', rq2['cliffs_delta'])
for pair, m in rq2['mcnemar'].items():
    print(f"McNemar {pair}: p={m['p_value']:.4f}, reject={m['reject_H0']}, achieved power~{m['achieved_power_approx']:.3f}")
verdict = 'REJECT H0' if any(m['reject_H0'] for m in rq2['mcnemar'].values()) else 'FAIL TO REJECT H0'
print(f"==> RQ2: {verdict} — no arm significantly outperforms at alpha=0.05 "
      f"(primary pooled McNemar, N={rq2['n_mutants_total']}); EvoMaster recall is numerically higher "
      '(0.135 vs 0.068) but p=0.064 > 0.05.')

overall recall: {'llm': 0.0677, 'manual': 0.0677, 'evomaster': 0.1353}
Friedman: chi2=0.0, p=1.0
Holm-corrected pairwise Wilcoxon: {
 "llm_vs_manual": {
  "p_raw": 1.0,
  "p_holm_adjusted": 1.0,
  "reject_H0": false
 },
 "llm_vs_evomaster": {
  "p_raw": 1.0,
  "p_holm_adjusted": 1.0,
  "reject_H0": false
 }
}
Cliff's delta: {'llm_vs_manual': 0.0, 'llm_vs_evomaster': -0.06766917293233082}
McNemar llm_vs_manual: p=1.0000, reject=False, achieved power~0.050
McNemar llm_vs_evomaster: p=0.0636, reject=False, achieved power~0.457
==> RQ2: FAIL TO REJECT H0 — no arm significantly outperforms at alpha=0.05 (primary pooled McNemar, N=133); EvoMaster recall is numerically higher (0.135 vs 0.068) but p=0.064 > 0.05.


In [5]:
pd.read_csv(os.path.join(os.path.dirname(RAW), 'stats', 'rq2_recall_per_sut.csv'))

,sut,evomaster,llm,manual
0,features,0.250000,0.250000,0.250000
1,ncs,0.171429,0.014286,0.014286
2,scs,0.084746,0.118644,0.118644


## RQ3 — Edge-case scenarios per endpoint (H1: LLM > Manual)

In [6]:
rq3 = s['RQ3_edge_cases_llm_vs_manual']
print(f"totals: LLM {rq3['llm_total_edge']} vs Manual {rq3['manual_total_edge']} "
      f"(median/op {rq3['llm_median_per_op']} vs {rq3['manual_median_per_op']})")
print(f"paired Wilcoxon (one-tailed): W={rq3['wilcoxon_W']}, p={rq3['p_value_one_sided']:.3e}")
print(f"effect size (rank-biserial) = {rq3['rank_biserial']:.3f}; "
      f"pairs LLM>Manual: {rq3['pairs_llm_more']}, Manual>LLM: {rq3['pairs_manual_more']}, ties: {rq3['pairs_tie']}")
verdict = 'REJECT H0' if rq3['reject_H0_(llm>manual)'] else 'FAIL TO REJECT H0'
print(f"==> RQ3: {verdict} — LLM produces more edge-case scenarios per endpoint (n={rq3['n_operations']})")

totals: LLM 217 vs Manual 141 (median/op 5.0 vs 4.0)
paired Wilcoxon (one-tailed): W=493.0, p=6.201e-07
effect size (rank-biserial) = 0.988; pairs LLM>Manual: 30, Manual>LLM: 1, ties: 4
==> RQ3: REJECT H0 — LLM produces more edge-case scenarios per endpoint (n=35)


## Multiplicity

In [7]:
print(json.dumps(s['multiplicity'], indent=1))

{
 "alpha_raw_per_RQ": 0.05,
 "alpha_bonferroni_3family": 0.0167,
 "note": "Per proposal.md \u00a74: Holm correction is applied WITHIN RQ2's post-hoc pairwise comparisons (see RQ2_fault_detection.pairwise_wilcoxon_holm); Bonferroni is applied ACROSS the 3 RQ families as a second, stricter reading (alpha_adj ~= 0.017). Both are reported; the per-RQ conclusions above use the raw alpha=0.05 threshold specified in proposal.md \u00a76.1, as pre-registered."
}


## Conclusions (per RQ, α=0.05 pre-registered)
| RQ | Verdict | Evidence |
|---|---|---|
| RQ1 | **Reject H0** | coverage 100% (35/35), p≈1.6e-9, rank-biserial 1.0 |
| RQ2 | **Fail to reject H0** | McNemar LLM-vs-EvoMaster p≈0.064; Cliff's δ≈−0.068 (negligible); achieved power ≈0.46 — underpowered, disclosed |
| RQ3 | **Reject H0** | LLM 217 vs Manual 141 edge-cases, p≈6.2e-7, rank-biserial 0.988 |

All conclusions also survive the Bonferroni cross-RQ reading (α_adj≈0.017) except none change: RQ1/RQ3 p-values ≪ 0.017; RQ2 remains non-significant.

**Note (full-run status):** these numbers reflect the current committed raw data. If LR regenerates the independent Manual suite (Week-8 bias fix) and re-runs the mutation matrix, re-run `scripts/compute_metric.py` then this notebook — RQ1 must remain byte-identical (LLM arm unchanged); RQ2/RQ3 Manual-side values are expected to change.